# MOEX × Chronos-2 — pipeline (raw zero-shot + fine-tuning)

End-to-end pipeline tailored for the MOEX hackathon experiment plan (see `wiki.md`).

**What this notebook does**
1. Sets up Chronos-2 in **Colab on a T4 GPU**.
2. Fetches MOEX OHLCV candles for a configurable group of tickers (default: 12 blue chips), plus index / FX / commodity covariates.
3. Brings everything onto a single regular timestamp grid (15m / 60m / 1d), fills market gaps, computes log-returns and the MVP covariate set from `wiki.md`.
4. Builds Chronos-2-ready inputs:
   - **Path A — raw zero-shot multivariate forecast** (group attention across all tickers, past-known market covariates + future-known calendar covariates).
   - **Path B — fine-tuning** Chronos-2 on the same panel via AutoGluon's `TimeSeriesPredictor`.

**Chronos-2 hard limits we respect** (from `huggingface.co/amazon/chronos-2`):
- Max context length: **8192 tokens**
- Max prediction length: **1024**
- 120M params, encoder-only, default dtype `float32` → ~0.48 GB weights; we cast to `bfloat16` on T4 → ~0.24 GB. Plenty of room on a 16 GB T4 even with a 12-series group at full context.
- Multivariate via `id_column` (group attention), past covariates as extra columns in `context_df`, future covariates via `future_df`.

**Sizing sanity check (CONTEXT_LEN per interval)**
- 1d: 250–500 bars → fits trivially.
- 60m: 300–1000 bars (~7h × 1000 = within 8192).
- 15m: 500–1500 bars → still well under 8192.

**Leakage rule (wiki §3):** market covariates are past-only (kept in `context_df`); calendar features are the only future-known signals (passed via `future_df`).


## 1. Colab + GPU setup

Run on a T4 runtime: **Runtime → Change runtime type → T4 GPU**.


In [ ]:
# Core deps. autogluon-timeseries is only needed for Path B (fine-tuning).
!pip install -q chronos-forecasting "pandas[pyarrow]" requests matplotlib numpy tqdm
# Uncomment for Path B (fine-tuning). Heavy install (~2-4 min on Colab).
# !pip install -q "autogluon.timeseries[chronos]"


In [ ]:
import os, math, json, warnings
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# GPU / dtype detection
HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"
# T4 supports fp16 well; bf16 is emulated on T4 (Turing) so prefer fp16 there.
if HAS_CUDA:
    cap = torch.cuda.get_device_capability(0)
    # Ampere+ (cap >= (8,0)) supports bf16 natively; Turing/T4 (7.5) does not.
    DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
    print(f"GPU: {torch.cuda.get_device_name(0)}  capability={cap}  dtype={DTYPE}")
else:
    DTYPE = torch.float32
    print("No CUDA — running on CPU (slow; demo only).")


## 2. Experiment config

Wiki-aligned defaults. Change `INTERVAL`, `TICKERS`, `HORIZON`, `CONTEXT_LEN` here — all downstream cells read from `CFG`.


In [ ]:
@dataclass
class Config:
    # --- universe (wiki §1) ---
    tickers: Sequence[str] = (
        # oil/gas/exporters + financial proxy
        "SBER", "GAZP", "LKOH", "ROSN", "NVTK", "TATN",
        # metals/miners
        "GMKN", "PLZL", "MAGN", "NLMK",
        # market/index proxy
        "MOEX", "VTBR",
    )
    # --- intervals (wiki §2) ---
    # MOEX ISS interval codes: 15 (15m), 60 (60m), 24 (1d)
    interval: int = 24            # one of: 15, 60, 24
    date_from: str = "2021-01-01"
    date_till: str = "2025-12-31"

    # --- horizon & context (wiki §5, §6) ---
    horizon: int = 5              # bars
    context_len: int = 250        # bars (must be <= 8192)

    # --- forecast quantiles (wiki §7) ---
    quantiles: tuple = (0.1, 0.5, 0.9)

    # --- index / commodity / FX covariates ---
    # MOEX index tickers (engine=stock, market=index)
    indexes: tuple = ("IMOEX", "MOEXOG", "MOEXMM", "MOEXFN", "RGBI")
    # FORTS futures used as proxies (engine=futures, market=forts)
    # BR = Brent, Si = USD/RUB, GD = Gold (PLZL)
    futures_proxies: tuple = ("BRH6", "SiH6", "GDH6")  # adjust to active contracts in your range

    # --- caches ---
    cache_dir: str = "./moex_cache"

CFG = Config()
os.makedirs(CFG.cache_dir, exist_ok=True)

# Sanity: respect Chronos-2 limits
assert CFG.context_len <= 8192, "Chronos-2 max context is 8192"
assert CFG.horizon <= 1024,     "Chronos-2 max prediction_length is 1024"
print(f"Config OK: {len(CFG.tickers)} tickers, interval={CFG.interval}, "
      f"context={CFG.context_len}, horizon={CFG.horizon}")


## 3. MOEX ISS fetchers (shares / indexes / FORTS)

Public delayed data — no auth, paginated 500 rows/request. We hit three engines:
- `stock/shares` — equities (TQBR by default)
- `stock/index` — index candles (IMOEX, MOEXOG, …)
- `futures/forts` — futures candles (BR, Si, GD)


In [ ]:
ISS_BASE = "https://iss.moex.com/iss"

def _iss_candles(engine: str, market: str, secid: str,
                 date_from: str, date_till: str, interval: int) -> pd.DataFrame:
    """Generic paginated ISS candles fetch."""
    url = f"{ISS_BASE}/engines/{engine}/markets/{market}/securities/{secid}/candles.json"
    rows, cols, start = [], None, 0
    while True:
        params = {"from": date_from, "till": date_till, "interval": interval, "start": start}
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()["candles"]
        cols = data["columns"]
        chunk = data["data"]
        if not chunk:
            break
        rows.extend(chunk)
        if len(chunk) < 500:
            break
        start += len(chunk)
    df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols or ["begin"])
    if not df.empty:
        df["begin"] = pd.to_datetime(df["begin"])
        df = df.sort_values("begin").reset_index(drop=True)
    return df

def fetch_share(secid, cfg=CFG):
    return _iss_candles("stock", "shares", secid, cfg.date_from, cfg.date_till, cfg.interval)

def fetch_index(secid, cfg=CFG):
    return _iss_candles("stock", "index", secid, cfg.date_from, cfg.date_till, cfg.interval)

def fetch_future(secid, cfg=CFG):
    return _iss_candles("futures", "forts", secid, cfg.date_from, cfg.date_till, cfg.interval)


def fetch_with_cache(fetcher, secid, cfg=CFG):
    """Parquet-cached wrapper — avoids re-hitting ISS on rerun."""
    fname = os.path.join(cfg.cache_dir, f"{fetcher.__name__}_{secid}_{cfg.interval}_{cfg.date_from}_{cfg.date_till}.parquet")
    if os.path.exists(fname):
        return pd.read_parquet(fname)
    df = fetcher(secid, cfg)
    if not df.empty:
        df.to_parquet(fname)
    return df


## 4. Timestamp regularisation, gap-fill, multivariate panel

Chronos-2 needs a **regular** timestamp index per series. MOEX has three irregularities:
- equities don't trade weekends and Russian holidays;
- intraday bars only exist during the trading session (main: 10:00–18:50 MSK = UTC+3);
- different tickers occasionally miss the same bar (suspensions, halts).

Strategy:
- 1d → `asfreq("B")` (business day) + ffill within the panel.
- 60m / 15m → build a custom session-aware grid then ffill.
- After per-series alignment we **inner-join across tickers** so all series share the exact same timestamp index — required for group attention.


In [ ]:
def _session_grid(start: pd.Timestamp, end: pd.Timestamp, interval_min: int) -> pd.DatetimeIndex:
    """Build MOEX main-session intraday grid (Mon-Fri, 10:00-18:50 MSK, naive)."""
    days = pd.bdate_range(start.normalize(), end.normalize())
    bars_per_day = (8 * 60 + 50) // interval_min  # 10:00 -> 18:50 = 530 min
    out = []
    for d in days:
        first = d + pd.Timedelta(hours=10)
        out.append(pd.date_range(first, periods=bars_per_day, freq=f"{interval_min}min"))
    return pd.DatetimeIndex(np.concatenate(out)) if out else pd.DatetimeIndex([])


def to_regular_series(df: pd.DataFrame, value_col: str, name: str,
                      interval: int = CFG.interval) -> pd.Series:
    """Reindex one ticker's column onto a regular grid; ffill across market gaps."""
    if df.empty:
        return pd.Series(name=name, dtype=float)
    s = df.set_index("begin")[value_col].astype(float).sort_index()
    if interval == 24:
        s = s.asfreq("B")
    else:
        grid = _session_grid(s.index.min(), s.index.max(), interval)
        s = s.reindex(grid)
    return s.ffill().rename(name)


def build_panel(prices_by_ticker: dict[str, pd.DataFrame],
                value_col: str = "close",
                interval: int = CFG.interval) -> pd.DataFrame:
    """Join all tickers on a shared regular index. Returns wide DataFrame."""
    cols = [to_regular_series(df, value_col, t, interval) for t, df in prices_by_ticker.items()]
    panel = pd.concat(cols, axis=1).dropna(how="any")  # inner-join on timestamps
    return panel


In [ ]:
def log_returns(panel: pd.DataFrame) -> pd.DataFrame:
    return np.log(panel / panel.shift(1)).dropna(how="any")


def calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    """Future-known covariates only (wiki §3 leakage rule)."""
    return pd.DataFrame({
        "hour": index.hour.astype(np.float32),
        "dow":  index.dayofweek.astype(np.float32),
        "dom":  index.day.astype(np.float32),
        "month": index.month.astype(np.float32),
    }, index=index)


def covariate_panel(prices_by_ticker, indexes_dict, futures_dict,
                    interval: int = CFG.interval) -> pd.DataFrame:
    """Build the past-only market covariate frame: index/FX/commodity log-returns + own volume."""
    parts = []
    # index returns
    for name, df in indexes_dict.items():
        s = to_regular_series(df, "close", f"{name}_close", interval)
        parts.append(np.log(s / s.shift(1)).rename(f"{name}_ret"))
    # futures (FX, brent, gold) returns
    for name, df in futures_dict.items():
        s = to_regular_series(df, "close", f"{name}_close", interval)
        parts.append(np.log(s / s.shift(1)).rename(f"{name}_ret"))
    # own volume change per ticker
    for tic, df in prices_by_ticker.items():
        v = to_regular_series(df, "volume", f"{tic}_vol", interval)
        parts.append(np.log1p(v).diff().rename(f"{tic}_dlogvol"))
    out = pd.concat(parts, axis=1)
    return out


## 5. Fetch and assemble the panel

Single cell pulls everything (cached). Subsequent reruns are instant.


In [ ]:
# 5a. tickers
prices = {}
for t in tqdm(CFG.tickers, desc="shares"):
    prices[t] = fetch_with_cache(fetch_share, t)

# 5b. indexes
indexes = {}
for t in tqdm(CFG.indexes, desc="indexes"):
    df = fetch_with_cache(fetch_index, t)
    if df.empty:
        print(f"  index {t}: empty (likely unsupported by ISS for this interval) — skipping")
        continue
    indexes[t] = df

# 5c. futures (proxies; if your range crosses contract rolls, adjust SECIDs accordingly)
futures = {}
for t in tqdm(CFG.futures_proxies, desc="futures"):
    df = fetch_with_cache(fetch_future, t)
    if df.empty:
        print(f"  future {t}: empty — try a different contract code (e.g. BRZ5, SiZ5, GDZ5)")
        continue
    futures[t] = df

print(f"Loaded: {len(prices)} tickers, {len(indexes)} indexes, {len(futures)} futures")


In [ ]:
# Build the wide panel of close prices and log-returns
price_panel = build_panel(prices, "close", CFG.interval)
ret_panel = log_returns(price_panel)
print(f"price panel: {price_panel.shape}   ret panel: {ret_panel.shape}")
print(f"date range:  {ret_panel.index.min()} → {ret_panel.index.max()}")
ret_panel.tail()


In [ ]:
# Past-only market covariates (shared across all tickers)
cov_panel = covariate_panel(prices, indexes, futures, CFG.interval)
cov_panel = cov_panel.reindex(ret_panel.index).ffill().dropna(how="any")
ret_panel = ret_panel.loc[cov_panel.index]    # realign
price_panel = price_panel.loc[cov_panel.index]
print(f"covariate panel: {cov_panel.shape}")
print(f"covariate cols ({cov_panel.shape[1]}): {list(cov_panel.columns)[:8]}...")
cov_panel.tail()


## 6. Path A — raw zero-shot multivariate Chronos-2

Single `predict_df` call with **all tickers as one group** (group attention) plus past market covariates and future calendar covariates.


In [ ]:
from chronos import Chronos2Pipeline

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=DEVICE,
    torch_dtype=DTYPE,
)
print("Chronos-2 loaded ✓")


In [ ]:
def build_chronos_inputs(ret_panel: pd.DataFrame,
                         cov_panel: pd.DataFrame,
                         tickers: Sequence[str],
                         context_len: int,
                         horizon: int):
    """
    Long-form context_df (one row per (id, timestamp)) and future_df with calendar covariates.
    target = log-return; past covariates = market covariates (broadcast across all ids);
    future covariates = calendar features (also broadcast).
    """
    # context = last `context_len` historical bars
    ctx_idx = ret_panel.index[-(context_len + horizon):-horizon]
    fut_idx = ret_panel.index[-horizon:]   # only used to get the timestamps

    # Past portion
    cov_ctx = cov_panel.loc[ctx_idx]
    cal_ctx = calendar_features(ctx_idx)
    rows = []
    for tic in tickers:
        block = pd.DataFrame({
            "id": tic,
            "timestamp": ctx_idx,
            "target": ret_panel[tic].loc[ctx_idx].values,
        })
        for c in cov_ctx.columns:
            block[c] = cov_ctx[c].values
        for c in cal_ctx.columns:
            block[c] = cal_ctx[c].values
        rows.append(block)
    context_df = pd.concat(rows, ignore_index=True)

    # Future = next `horizon` bars; only future-known (calendar) covariates carried.
    cal_fut = calendar_features(fut_idx)
    fut_rows = []
    for tic in tickers:
        block = pd.DataFrame({"id": tic, "timestamp": fut_idx})
        for c in cal_fut.columns:
            block[c] = cal_fut[c].values
        fut_rows.append(block)
    future_df = pd.concat(fut_rows, ignore_index=True)

    return context_df, future_df, fut_idx

context_df, future_df, fut_idx = build_chronos_inputs(
    ret_panel, cov_panel, CFG.tickers, CFG.context_len, CFG.horizon
)
print(f"context_df: {context_df.shape}   future_df: {future_df.shape}")
print(f"forecast window: {fut_idx[0]} → {fut_idx[-1]}")
context_df.head()


In [ ]:
# Single multivariate predict call — group attention runs across all tickers at once.
pred_df = pipeline.predict_df(
    context_df,
    future_df=future_df,
    prediction_length=CFG.horizon,
    quantile_levels=list(CFG.quantiles),
    id_column="id",
    timestamp_column="timestamp",
    target="target",
)
print(f"pred_df: {pred_df.shape}")
pred_df.head()


In [ ]:
# Per-ticker metrics: directional accuracy on returns, MAE on prices.
def evaluate_zero_shot(pred_df, ret_panel, price_panel, fut_idx, tickers, quantiles):
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].sort_values("timestamp")
        y_true_ret = ret_panel[tic].loc[fut_idx].values
        y_pred_ret = sub[str(quantiles[len(quantiles)//2])].values  # median
        # reconstruct prices for MAE/MAPE
        last_p = price_panel[tic].loc[fut_idx[0]] / np.exp(y_true_ret[0])  # price just before fut_idx[0]
        y_pred_p = last_p * np.exp(np.cumsum(y_pred_ret))
        y_true_p = price_panel[tic].loc[fut_idx].values
        rows.append({
            "ticker": tic,
            "dir_acc_%": float((np.sign(y_pred_ret) == np.sign(y_true_ret)).mean() * 100),
            "ret_corr":  float(np.corrcoef(y_pred_ret, y_true_ret)[0, 1]) if len(y_true_ret) > 1 else np.nan,
            "mae_price": float(np.mean(np.abs(y_true_p - y_pred_p))),
            "mape_%":    float(np.mean(np.abs((y_true_p - y_pred_p) / y_true_p)) * 100),
        })
    return pd.DataFrame(rows)

eval_df = evaluate_zero_shot(pred_df, ret_panel, price_panel, fut_idx, CFG.tickers, CFG.quantiles)
print(eval_df.to_string(index=False))
print(f"\nmean DirAcc: {eval_df['dir_acc_%'].mean():.1f}%   mean ret_corr: {eval_df['ret_corr'].mean():+.3f}")


In [ ]:
# Quick visualisation: 4 tickers in a grid
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
median_q = str(CFG.quantiles[len(CFG.quantiles)//2])
lo_q, hi_q = str(CFG.quantiles[0]), str(CFG.quantiles[-1])

for ax, tic in zip(axes.flat, CFG.tickers[:4]):
    sub = pred_df[pred_df["id"] == tic].sort_values("timestamp")
    hist_idx = ret_panel.index[-(60 + CFG.horizon):-CFG.horizon]
    hist_p = price_panel[tic].loc[hist_idx]
    last_p = hist_p.iloc[-1]
    p_med = last_p * np.exp(np.cumsum(sub[median_q].values))
    p_lo  = last_p * np.exp(np.cumsum(sub[lo_q].values))
    p_hi  = last_p * np.exp(np.cumsum(sub[hi_q].values))
    truth = price_panel[tic].loc[fut_idx]

    ax.plot(hist_idx, hist_p.values, color="#444", lw=1)
    ax.plot(fut_idx, truth.values, "g.-", label="actual")
    ax.plot(fut_idx, p_med, "r.-", label="forecast median")
    ax.fill_between(fut_idx, p_lo, p_hi, color="red", alpha=0.15, label=f"q[{lo_q},{hi_q}]")
    ax.axvline(fut_idx[0], ls="--", color="gray", lw=1)
    ax.set_title(tic); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(f"Chronos-2 zero-shot multivariate ({len(CFG.tickers)} tickers as group)", fontsize=12)
plt.tight_layout(); plt.show()


## 7. Path B — fine-tuning Chronos-2 (AutoGluon)

The supported fine-tuning path for Chronos-2 today is **AutoGluon's `TimeSeriesPredictor`** with `model="Chronos"` / `"Chronos[chronos-2]"`. It handles the data wrapping, optimizer, and validation loop.

**Heads-up:**
- Install is heavy (~2-4 min): uncomment the `pip install` in §1.
- On a T4, a small fine-tune (a few hundred steps) on a 12-series panel completes in single-digit minutes; longer runs are fine but you'll want to set `time_limit` explicitly.
- We hold out the **last `horizon` bars** as validation.


In [ ]:
# Uncomment and run after the autogluon install in §1 succeeded.
# from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


In [ ]:
def to_ag_long(ret_panel: pd.DataFrame,
               cov_panel: pd.DataFrame,
               tickers: Sequence[str]) -> pd.DataFrame:
    """Build AutoGluon-friendly long-form: item_id, timestamp, target, + past covariates."""
    blocks = []
    for tic in tickers:
        df = pd.DataFrame({
            "item_id": tic,
            "timestamp": ret_panel.index,
            "target":   ret_panel[tic].values,
        })
        for c in cov_panel.columns:
            df[c] = cov_panel[c].values
        blocks.append(df)
    return pd.concat(blocks, ignore_index=True)

ag_long = to_ag_long(ret_panel, cov_panel, CFG.tickers)
print(f"ag_long: {ag_long.shape}   sample cols: {list(ag_long.columns)[:6]}")
ag_long.head()


In [ ]:
# Run this cell only after `from autogluon.timeseries import ...` succeeds.
#
# ts_df = TimeSeriesDataFrame.from_data_frame(
#     ag_long,
#     id_column="item_id",
#     timestamp_column="timestamp",
# )
#
# predictor = TimeSeriesPredictor(
#     prediction_length=CFG.horizon,
#     target="target",
#     known_covariates_names=[],         # calendar covariates can be added here if pre-computed
#     quantile_levels=list(CFG.quantiles),
#     eval_metric="WQL",
#     path="./ag_chronos2_ft",
# )
#
# predictor.fit(
#     ts_df,
#     hyperparameters={
#         "Chronos": {
#             "model_path": "amazon/chronos-2",
#             "fine_tune": True,
#             "fine_tune_steps": 500,        # bump for real runs; T4 ~1-2 min/100 steps
#             "fine_tune_lr": 1e-5,
#             "context_length": CFG.context_len,
#             "torch_dtype": "bfloat16" if DTYPE == torch.bfloat16 else "float16",
#             "device": DEVICE,
#         }
#     },
#     time_limit=900,                        # 15 min budget — adjust for your run
#     enable_ensemble=False,
# )
#
# predictor.leaderboard(ts_df)


In [ ]:
# Inference with the fine-tuned predictor — same shape as Path A predictions.
#
# ag_pred = predictor.predict(ts_df)
# print(ag_pred.head())
#
# # Convert back to wide form per ticker if needed:
# ft_pred_df = ag_pred.reset_index().rename(columns={"item_id": "id"})


## 8. What to vary next (mapped to the wiki)

- **Universe:** swap `CFG.tickers` for a sector subset to test whether group attention pays off more inside one sector.
- **Interval:** flip `CFG.interval` to `60` or `15` and rerun §5–§6. The session grid + ffill in §4 already handle intraday gaps; the Chronos-2 context cap (8192) is enforced by the assert in §2.
- **Horizon grid (wiki §5):** loop §6 with `CFG.horizon ∈ {1, 2, 4}` (intraday) or `{3, 5, 10}` (daily).
- **Walk-forward (wiki §6):** wrap §6 in a sliding-window loop (shift = horizon for intraday, 1 bar for daily).
- **Baselines (wiki §7):** add zero-return / last-return / CatBoost predictions on the same `ret_panel` for a like-for-like comparison.
